# Detección de Virus Troyanos con Redes Neuronales Convolucionales Sobre Grafos (GCN)

## Introducción 

Los **virus troyanos** son una de las amenazas más comunes y peligrosas en ciberseguridad. Se caracterizan por disfrazarse como software legítimo para engañar al usuario y ejecutar acciones maliciosas, como robo de información, control remoto o instalación de otros tipos de malware.

Tradicionalmente, la detección de troyanos se ha basado en firmas o reglas específicas. Sin embargo, este enfoque es **limitado frente a variantes nuevas o mutaciones** del malware, ya que no generaliza bien ante ataques desconocidos.

## Solución basada en Machine Learning

El aprendizaje automático ofrece una alternativa poderosa: entrenar modelos capaces de **identificar patrones anómalos** en los datos, incluso si el virus es desconocido. Pero la mayoría de estos modelos trabajan con datos tabulares o secuencias, lo cual puede ser insuficiente para capturar **relaciones estructurales** más complejas.

## Redes Convolucionales sobre Grafos (GCN)

Las **Graph Convolutional Networks (GCNs)** permiten modelar relaciones entre instancias como un grafo, donde:
- Cada **nodo** representa una muestra (por ejemplo, un paquete de red).
- Cada **arista** representa una relación de similitud (por ejemplo, cercanía en características).

Este enfoque permite capturar **información contextual** entre muestras, mejorando la capacidad del modelo para detectar troyanos sutiles o camuflados.

![Graph Convolutional Networks](https://snap.stanford.edu/decagon/decagon-overview.png)


In [5]:
# 1. Imports básicos
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import torch_geometric

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from torch import nn
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.loader import DataLoader

In [6]:
print(torch.__version__)
print(torch.version.cuda)
print(torch_geometric.__version__)
print(torch.cuda.is_available())

2.7.1+cu128
12.8
2.6.1
True


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [10]:
BASE_DIR = Path().cwd().parent
DATA_DIR = BASE_DIR / 'assets'

In [11]:
## 2. Carga y Exploración del dataset
# Ruta directa al CSV ya cargado en Colab
data_path = DATA_DIR / 'Trojan_Detection.csv'

# Leer CSV principal
df = pd.read_csv(data_path)

display(df.head())
print("Columnas en el dataset:\n", df.columns.tolist())

,Unnamed: 0,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Class
0,73217,10.42.0.42-121.14.255.84-49975-80-6,10.42.0.42,49975,121.14.255.84,80,6,17/07/2017 01:18:33,10743584,4,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Trojan
1,72089,172.217.6.226-10.42.0.42-443-49169-17,10.42.0.42,49169,172.217.6.226,443,17,17/07/2017 10:25:25,254217,6,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Trojan
2,96676,10.42.0.1-10.42.0.42-53-37749-17,10.42.0.42,37749,10.42.0.1,53,17,30/06/2017 07:16:12,1023244,1,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
3,42891,10.42.0.1-10.42.0.42-53-41352-17,10.42.0.42,41352,10.42.0.1,53,17,13/07/2017 03:48:44,286483,1,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Trojan
4,169326,10.42.0.151-107.22.241.77-44353-443-6,10.42.0.151,44353,107.22.241.77,443,6,05/07/2017 10:47:35,65633087,12,...,32,322594.0,0.0,322594.0,322594.0,60306983.0,0.0,60306983.0,60306983.0,Benign


Columnas en el dataset:
 ['Unnamed: 0', 'Flow ID', ' Source IP', ' Source Port', ' Destination IP', ' Destination Port', ' Protocol', ' Timestamp', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Pack

In [12]:

print("Distribución de clases:\n", df['Class'].value_counts())

Distribución de clases:
 Class
Trojan    90683
Benign    86799
Name: count, dtype: int64


In [13]:
## 3. Preprocesamiento de Datos

# Mapear etiquetas: Benign=0, Trojan=1
y = df['Class'].map({'Benign': 0, 'Trojan': 1}).values

# Seleccionar solo features numéricos
num_df = df.select_dtypes(include=[np.number]).copy()
# Eliminar columna de índice si existe
if 'Unnamed: 0' in num_df.columns:
    num_df = num_df.drop(columns=['Unnamed: 0'])
X = num_df.values

# Normalizar features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Dividir en training/validation
tX, vX, ty, vy = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

In [14]:
## 4. Construcción del Grafo con KNN

k = 8  # número de vecinos
nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='ball_tree').fit(tX)
distances, indices = nbrs.kneighbors(tX)

# Generar edge_index bidireccional
src, dst = [], []
for i, neigh in enumerate(indices):
    for j in neigh[1:]:  # omitir self-loop
        src.extend([i, j])
        dst.extend([j, i])
edge_index = torch.tensor([src, dst], dtype=torch.long)

# Crear objeto PyG Data para entrenamiento
data = Data(
    x=torch.tensor(tX, dtype=torch.float),
    edge_index=edge_index,
    y=torch.tensor(ty, dtype=torch.long)
)
loader = DataLoader([data], batch_size=1)

In [ ]:
# Construcción del grafo para validación con KNN
nbrs_val = NearestNeighbors(n_neighbors=k+1, algorithm='ball_tree').fit(vX)
distances_val, indices_val = nbrs_val.kneighbors(vX)

src_val, dst_val = [], []
for i, neigh in enumerate(indices_val):
    for j in neigh[1:]:  # omitir self-loop
        src_val.extend([i, j])
        dst_val.extend([j, i])
edge_index_val = torch.tensor([src_val, dst_val], dtype=torch.long)

# Crear objeto Data para validación
val_data = Data(
    x=torch.tensor(vX, dtype=torch.float),
    edge_index=edge_index_val,
    y=torch.tensor(vy, dtype=torch.long)
)

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.out = nn.Linear(hidden_channels, 1)
        self.dropout = nn.Dropout(dropout)


    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        return self.out(x)
        # return F.log_softmax(x, dim=1)


In [ ]:
# Instanciar modelo (en CPU para evitar errores CUDA)
model_gcn = GCN(
    in_channels=tX.shape[1],
    hidden_channels=32,
    dropout=0.5
).to(device)

model_gcn

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_gcn.parameters(), lr=0.01, weight_decay=5e-4)

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

X_train = data.x.to(device)
edge_index_train = data.edge_index.to(device)

epochs = 500
for epoch in range(epochs):
    model_gcn.train()

    y_logits = model_gcn(X_train, edge_index_train)
    y_pred = torch.round(torch.sigmoid(y_logits))

    loss = loss_fn(y_logits, data.y.to(device).float().view(-1, 1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model_gcn.eval()
    with torch.no_grad():
        X_val = val_data.x.to(device)
        edge_index_val = val_data.edge_index.to(device)

        y_logits_val = model_gcn(X_val, edge_index_val)
        y_pred_val = torch.round(torch.sigmoid(y_logits_val))

        val_loss = loss_fn(y_logits_val, val_data.y.to(device).float().view(-1, 1))
        val_acc = (y_pred_val == val_data.y.to(device)).float().mean()

    if epoch % 50 == 0:
        print(f"Epoch {epoch:03d}, Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")


## Mejorando el Modelo

In [ ]:
print(np.unique(ty, return_counts=True))
print(pd.DataFrame(tX).describe())